# UniProt Fetch Examples

This notebook demonstrates protein entry retrieval from UniProt using **`run_uniprot_fetch`**.

- **Direct lookup** -- Fetch an entry by UniProt accession
- **Name search** -- Search by gene name and organism with ranked selection
- **Cross-references** -- Extract PDB structure links from UniProt entries

## Imports

In [1]:
from bio_programming_tools import UniProtFetchConfig, UniProtFetchInput, run_uniprot_fetch


## API Reference

**`UniProtFetchInput`**

| Field | Type | Default | Description |
|---|---|---|---|
| `uniprot_id` | `Optional[str]` | `None` | UniProt accession for direct lookup |
| `target_name` | `Optional[str]` | `None` | Gene or protein name for search |
| `organism` | `Optional[str]` | `None` | Organism for search disambiguation |
| `prefer_pdb_crossref` | `bool` | `False` | When searching, prefer entries that have linked PDB structures |
| `max_candidates` | `int` | `5` | Maximum number of search results to evaluate when ranking |

**`UniProtFetchConfig`**

| Field | Type | Default | Description |
|---|---|---|---|
| `request_timeout_seconds` | `int` | `15` | HTTP timeout in seconds |
| `http_retries` | `int` | `2` | Retries for HTTP requests |
| `backoff_seconds` | `float` | `1.0` | Seconds to wait between retries (doubles after each attempt) |
| `user_agent` | `str` | `"bio-programming-tools/uniprot-fetch-v1"` | Identifier string sent to database APIs with each request |

**`UniProtFetchOutput`**

| Field | Type | Description |
|---|---|---|
| `accession` | `str` | Primary UniProt accession |
| `sequence` | `Optional[str]` | Protein sequence |
| `length` | `Optional[int]` | Sequence length |
| `entry_type` | `Optional[str]` | Review status (e.g. `UniProtKB reviewed (Swiss-Prot)` for curated entries) |
| `gene_names` | `List[str]` | Gene name symbols |
| `pdb_crossrefs` | `List[str]` | PDB structure IDs linked to this protein entry |
| `source_url` | `str` | UniProt entry URL |
| `raw_entry` | `Dict[str, Any]` | Complete UniProt JSON record for advanced programmatic access |

## 1. Fetch by UniProt Accession

Retrieve the TP53 protein entry directly by its UniProt accession.

In [2]:
inputs = UniProtFetchInput(uniprot_id="P04637")

output = run_uniprot_fetch(inputs, UniProtFetchConfig())

print(f"Accession: {output.accession}")
print(f"Entry type: {output.entry_type}")
print(f"Gene names: {output.gene_names}")
print(f"Sequence length: {output.length}")
print(f"Preview: {output.sequence[:60]}...")
print(f"PDB cross-refs: {output.pdb_crossrefs[:5]} ({len(output.pdb_crossrefs)} total)")
print(f"Source: {output.source_url}")

Accession: P04637
Entry type: UniProtKB reviewed (Swiss-Prot)
Gene names: ['tp53']
Sequence length: 393
Preview: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...
PDB cross-refs: ['1A1U', '1AIE', '1C26', '1DT7', '1GZH'] (286 total)
Source: https://rest.uniprot.org/uniprotkb/P04637


## 2. Search by Gene Name and Organism

When you don't have a UniProt accession, search by gene name and organism. The tool ranks candidates by gene name match, reviewed status, and optionally PDB availability.

In [3]:
inputs = UniProtFetchInput(
    target_name="lacI",
    organism="Escherichia coli",
)

output = run_uniprot_fetch(inputs, UniProtFetchConfig())

print(f"Accession: {output.accession}")
print(f"Gene names: {output.gene_names}")
print(f"Sequence length: {output.length}")
print(f"Preview: {output.sequence[:60]}...")

Accession: A0A094XSW9
Gene names: ['laci']
Sequence length: 319
Preview: MAELNYIPNRVAQQLAGKQSLLIGVATSSLALHAPSQIVAAIKSRADQLGASVVVSMVER...


## 3. Search with PDB Cross-Reference Preference

Set `prefer_pdb_crossref=True` to prioritize entries that have linked PDB structures. Useful when you need both protein sequence and structure.

In [4]:
inputs = UniProtFetchInput(
    target_name="insulin",
    organism="Homo sapiens",
    prefer_pdb_crossref=True,
    max_candidates=10,
)

output = run_uniprot_fetch(inputs, UniProtFetchConfig())

print(f"Accession: {output.accession}")
print(f"Gene names: {output.gene_names}")
print(f"Sequence length: {output.length}")
print(f"PDB cross-refs: {output.pdb_crossrefs[:5]}..." if len(output.pdb_crossrefs) > 5 else f"PDB cross-refs: {output.pdb_crossrefs}")
print(f"Total PDB links: {len(output.pdb_crossrefs)}")

Accession: P01308
Gene names: ['ins']
Sequence length: 110
PDB cross-refs: ['1A7F', '1AI0', '1AIY', '1B9E', '1BEN']...
Total PDB links: 367


## Export Results

Save fetched results to JSON format.

In [5]:
# Export UniProt result to JSON
output.export("uniprot_results", export_path="./example_output", file_format="json")
print("Exported to ./example_output/uniprot_results.json")

Exported to ./example_output/uniprot_results.json
